# Notebook 04 — SQL Database & 50 Analytical Business Queries

**Project:** Enterprise E-Commerce Operations and Customer Experience Control Tower  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Phase:** 6 — SQL Database & Analytical Query Suite  

## Objectives
- Connect to the enterprise SQLite Database (`data/processed/ecommerce_control_tower.db`)
- Verify Star Schema facts and dimension tables
- Execute 50 production-grade SQL queries across 6 business domains:
  1. Executive & Revenue KPIs
  2. Delivery & Operational SLA KPIs
  3. Customer Experience & CSAT KPIs
  4. Seller & Category Performance Scorecards
  5. Customer Retention & Payment Analytics
  6. Advanced Analytics: CTEs, Window Functions (LAG, DENSE_RANK, Rolling Averages, Pareto 80/20)
- Validate financial reconciliation across Python and SQL

---
## 0. Setup & Database Connection

In [ ]:
import sqlite3
import pandas as pd
import os

DB_PATH = '../data/processed/ecommerce_control_tower.db'
conn = sqlite3.connect(DB_PATH)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Check tables in SQLite Database
tables = pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'index') ORDER BY type, name;", conn)
print(f'Connected to SQLite DB: {DB_PATH}')
tables.head(15)

---
## 1. Executive & Revenue KPIs (Queries 1–10)

In [ ]:
# Query 4: Total GMV, Freight, and Order Value
q_exec = """
SELECT 
    COUNT(order_id) AS total_orders,
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    ROUND(SUM(gmv), 2) AS total_gmv_brl,
    ROUND(SUM(freight_value), 2) AS total_freight_brl,
    ROUND(SUM(total_order_value), 2) AS total_gross_order_value_brl,
    ROUND(AVG(gmv), 2) AS aov_brl
FROM fact_orders;
"""
pd.read_sql_query(q_exec, conn)

In [ ]:
# Query 10: Monthly GMV, Orders, and Freight Trajectory
q_monthly = """
SELECT 
    SUBSTR(order_purchase_timestamp, 1, 7) AS order_year_month,
    COUNT(order_id) AS orders,
    ROUND(SUM(gmv), 2) AS monthly_gmv,
    ROUND(SUM(freight_value), 2) AS monthly_freight,
    ROUND(SUM(gmv) / COUNT(order_id), 2) AS monthly_aov
FROM fact_orders
WHERE order_purchase_timestamp IS NOT NULL
GROUP BY SUBSTR(order_purchase_timestamp, 1, 7)
ORDER BY order_year_month;
"""
pd.read_sql_query(q_monthly, conn).tail(12)

---
## 2. Delivery & Operational SLA KPIs (Queries 11–20)

In [ ]:
# Query 11 & 12: On-Time Delivery vs Delays & Severe Delays
q_sla = """
SELECT 
    COUNT(*) AS total_delivered_orders,
    SUM(CASE WHEN late_delivery_flag = 0 THEN 1 ELSE 0 END) AS on_time_orders,
    ROUND(100.0 * SUM(CASE WHEN late_delivery_flag = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS on_time_rate_pct,
    SUM(late_delivery_flag) AS late_orders,
    ROUND(100.0 * SUM(late_delivery_flag) / COUNT(*), 2) AS late_rate_pct,
    SUM(severe_delay_flag) AS severe_delay_orders_gt_7d,
    ROUND(100.0 * SUM(severe_delay_flag) / COUNT(*), 2) AS severe_delay_rate_pct
FROM fact_orders
WHERE order_status = 'delivered' 
  AND order_delivered_customer_date IS NOT NULL;
"""
pd.read_sql_query(q_sla, conn)

In [ ]:
# Query 15: State-Level Delivery SLA & Delay Performance
q_state_delivery = """
SELECT 
    customer_state,
    COUNT(order_id) AS delivered_orders,
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
    ROUND(AVG(delay_days), 2) AS avg_delay_days,
    ROUND(100.0 * (1.0 - 1.0 * SUM(late_delivery_flag) / COUNT(*)), 2) AS on_time_rate_pct
FROM fact_orders
WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
GROUP BY customer_state
ORDER BY delivered_orders DESC
LIMIT 10;
"""
pd.read_sql_query(q_state_delivery, conn)

---
## 3. Customer Experience & CSAT KPIs (Queries 21–28)

In [ ]:
# Query 24: Direct Correlation Between Delivery Performance & Review Scores
q_csat_delivery = """
SELECT 
    CASE WHEN late_delivery_flag = 1 THEN 'Late Delivery' ELSE 'On-Time / Early' END AS delivery_performance,
    COUNT(order_id) AS order_count,
    ROUND(AVG(review_score_avg), 2) AS avg_review_score,
    ROUND(100.0 * SUM(low_review_flag) / COUNT(*), 2) AS negative_review_rate_pct,
    ROUND(100.0 * SUM(high_review_flag) / COUNT(*), 2) AS positive_review_rate_pct
FROM fact_orders
WHERE order_status = 'delivered' AND review_score_avg IS NOT NULL
GROUP BY late_delivery_flag;
"""
pd.read_sql_query(q_csat_delivery, conn)

---
## 4. Seller & Category Performance (Queries 29–36)

In [ ]:
# Query 32: High-GMV Poor-Service Sellers (Operational Risk Quadrant)
q_risk_sellers = """
SELECT 
    seller_id,
    seller_state,
    total_gmv,
    total_orders,
    avg_review_score,
    late_delivery_rate
FROM dim_sellers
WHERE total_gmv >= 20000 
  AND avg_review_score < 3.8
ORDER BY total_gmv DESC;
"""
pd.read_sql_query(q_risk_sellers, conn).head(10)

---
## 5. Advanced SQL: CTEs, Window Functions & Pareto (Queries 43–50)

In [ ]:
# Query 43: Seller Pareto 80/20 Cumulative Contribution
q_pareto = """
WITH SellerRevenue AS (
    SELECT 
        seller_id,
        total_gmv,
        SUM(total_gmv) OVER () AS platform_gmv,
        SUM(total_gmv) OVER (ORDER BY total_gmv DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_gmv,
        ROW_NUMBER() OVER (ORDER BY total_gmv DESC) AS seller_rank,
        COUNT(*) OVER () AS total_sellers
    FROM dim_sellers
)
SELECT 
    seller_rank,
    seller_id,
    total_gmv,
    ROUND(100.0 * cumulative_gmv / platform_gmv, 2) AS cumulative_gmv_pct,
    ROUND(100.0 * seller_rank / total_sellers, 2) AS pct_of_sellers
FROM SellerRevenue
WHERE seller_rank IN (10, 50, 100, 200, 500, 1000, 2000, 3095);
"""
pd.read_sql_query(q_pareto, conn)

In [ ]:
# Query 45: Month-over-Month (MoM) GMV Growth with LAG()
q_mom = """
WITH MonthlyStats AS (
    SELECT 
        SUBSTR(order_purchase_timestamp, 1, 7) AS ym,
        COUNT(order_id) AS orders,
        SUM(gmv) AS gmv
    FROM fact_orders
    WHERE order_purchase_timestamp IS NOT NULL
    GROUP BY SUBSTR(order_purchase_timestamp, 1, 7)
)
SELECT 
    ym,
    orders,
    ROUND(gmv, 2) AS current_month_gmv,
    ROUND(LAG(gmv, 1) OVER (ORDER BY ym), 2) AS previous_month_gmv,
    ROUND(100.0 * (gmv - LAG(gmv, 1) OVER (ORDER BY ym)) / LAG(gmv, 1) OVER (ORDER BY ym), 2) AS mom_growth_pct
FROM MonthlyStats
ORDER BY ym;
"""
pd.read_sql_query(q_mom, conn).tail(12)

In [ ]:
# Query 50: Pipeline Integrity & Financial Reconciliation
q_reconcile = """
SELECT 
    (SELECT COUNT(*) FROM fact_orders) AS total_fact_orders,
    (SELECT COUNT(*) FROM fact_order_items) AS total_fact_order_items,
    (SELECT COUNT(*) FROM dim_customers) AS total_dim_customers,
    (SELECT COUNT(*) FROM dim_sellers) AS total_dim_sellers,
    (SELECT ROUND(SUM(gmv), 2) FROM fact_orders) AS fact_orders_total_gmv,
    (SELECT ROUND(SUM(price), 2) FROM fact_order_items) AS fact_items_total_price_gmv,
    (SELECT ROUND(SUM(total_gmv), 2) FROM dim_sellers) AS dim_sellers_total_gmv;
"""
pd.read_sql_query(q_reconcile, conn)